# Homework 1

# Классификация на PyTorch Lightning


In [ ]:
# Установка зависимостей
%pip install -r requirements.txt

In [12]:
import torch
import torch.nn as nn
from torch.optim import Optimizer
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from typing import Any, Dict, List, Optional, Union

In [13]:
wine = load_wine()
X = wine.data
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Classes: {len(np.unique(y))}")


Train: torch.Size([142, 13]), Test: torch.Size([36, 13])
Classes: 3


In [14]:
class WineDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class WineDataModule(pl.LightningDataModule):
    def __init__(self, X_train: Any, X_test: Any, y_train: Any, y_test: Any, batch_size=16) -> None:
        

        """
        Инициализация модуля данных для датасета вина.
        
        Args:
            X_train: Тренировочные признаки
            X_test: Тестовые признаки
            y_train: Тренировочные метки
            y_test: Тестовые метки
            batch_size: Размер батча
        """
        super().__init__()

        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.batch_size = batch_size
    
    def setup(self, stage: Optional[str] = None) -> None:
        """
        Настройка датасетов для различных стадий (train/val/test).
        
        Args:
            stage: Стадия настройки ('fit', 'validate', 'test', 'predict' или None)
        """

        self.train_dataset = WineDataset(self.X_train, self.y_train)
        self.val_dataset = WineDataset(self.X_test, self.y_test)
        self.test_dataset = WineDataset(self.X_test, self.y_test)
    
    def train_dataloader(self) -> DataLoader:
        """
        Загрузчик данных для обучения.
        
        Returns:
            DataLoader: Загрузчик тренировочных данных
        """

        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)
    
    def val_dataloader(self) -> DataLoader:
        """
        Загрузчик данных для валидации.
        
        Returns:
            DataLoader: Загрузчик валидационных данных
        """

        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)
    
    def test_dataloader(self) -> DataLoader:
        """
        Загрузчик данных для тестирования.
        
        Returns:
            DataLoader: Загрузчик тестовых данных
        """

        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False)

In [15]:
class NeuralNetLightning(pl.LightningModule):
    def __init__(self, input_size: int, num_classes: int, lr: float = 0.001) -> None:
        """
        Инициализация нейронной сети с использованием PyTorch Lightning.
        
        Args:
            input_size: Размер входных признаков
            num_classes: Количество классов для классификации
            lr: Learning rate для оптимизатора
        """
        
        
        super().__init__()
        # Сохраняем гиперпараметры (lr, input_size, num_classes)
        # Они будут доступны через self.hparams
        self.save_hyperparameters()
        
        # Определяем архитектуру сети
        hidden_sizes = [32, 16] 
        layers = []
        prev_size = self.hparams.input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, self.hparams.num_classes))
        
        self.network = nn.Sequential(*layers)
        
        # Функция потерь
        self.criterion = nn.CrossEntropyLoss()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Прямой проход через нейронную сеть.
        
        Args:
            x: Входной тензор
            
        Returns:
            Выходной тензор сети
        """

        return self.network(x)
    
    def training_step(self, batch: Any, batch_idx: int) -> torch.Tensor:
        """
        Шаг обучения.
        
        Args:
            batch: Батч данных
            batch_idx: Индекс батча
            
        Returns:
            Значение функции потерь
        """

        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        
        # Логгирование (для отслеживания)
        self.log('train_loss', loss)
        return loss
    
    def validation_step(self, batch: Any, batch_idx: int) -> torch.Tensor:
        """
        Шаг валидации (вызывается во время trainer.fit).
        
        Args:
            batch: Батч данных
            batch_idx: Индекс батча
            
        Returns:
            Значение функции потерь
        """

        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        preds = torch.argmax(y_hat, dim=1)
        
        # accuracy_score из sklearn
        acc = accuracy_score(y.cpu(), preds.cpu()) 
        
        # Логгируем и выводим в progress bar
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)
        return loss
    
    def test_step(self, batch: Any, batch_idx: int) -> torch.Tensor:
        """
        Шаг тестирования (вызывается во время trainer.test).
        
        Args:
            batch: Батч данных
            batch_idx: Индекс батча
            
        Returns:
            Значение функции потерь
        """

        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        preds = torch.argmax(y_hat, dim=1)
        
        acc = accuracy_score(y.cpu(), preds.cpu())
        
        # Логгируем метрики теста
        self.log('test_loss', loss)
        self.log('test_acc', acc)
        return loss
    
    def configure_optimizers(self) -> Optimizer:
        """
        Настройка оптимизатора (Adam).
        
        Returns:
            Оптимизатор для обучения
        """
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        return optimizer

In [16]:
data_module = WineDataModule(X_train, X_test, y_train, y_test, batch_size=16)
model = NeuralNetLightning(input_size=13, num_classes=3, lr=0.001)

print(model)


NeuralNetLightning(
  (network): Sequential(
    (0): Linear(in_features=13, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=3, bias=True)
  )
  (criterion): CrossEntropyLoss()
)


In [17]:
trainer = pl.Trainer(
    max_epochs=100,
    accelerator='auto',
    devices=1,
    log_every_n_steps=5,
    enable_progress_bar=True
)

trainer.fit(model, data_module)


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | network   | Sequential       | 1.0 K  | train
1 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
1.0 K     Trainable params
0         Non-trainable params
1.0 K     Total params
0.004     Total estimated model params size (MB)
7         Modules in train mode
0         Modules in eval mode


c:\_MyGit\Deep_learning_autumn_2025\Deep_learning_MISIS\hwvenv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
c:\_MyGit\Deep_learning_autumn_2025\Deep_learning_MISIS\hwvenv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 99: 100%|██████████| 9/9 [00:00<00:00, 61.55it/s, v_num=2, val_loss=0.00116, val_acc=1.000]  

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 9/9 [00:00<00:00, 61.55it/s, v_num=2, val_loss=0.00116, val_acc=1.000]


In [18]:
test_result = trainer.test(model, data_module)
print(f"Test Accuracy: {test_result[0]['test_acc']:.4f}")


c:\_MyGit\Deep_learning_autumn_2025\Deep_learning_MISIS\hwvenv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 3/3 [00:00<00:00, 120.60it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc                    1.0
        test_loss          0.0011609657667577267
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Test Accuracy: 1.0000


In [19]:
model.eval()
with torch.no_grad():
    logits = model(X_test)
    predicted = torch.argmax(logits, dim=1)
    
from sklearn.metrics import classification_report
print("\nClassification Report:")
print(classification_report(y_test, predicted, target_names=wine.target_names))



Classification Report:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        14
     class_1       1.00      1.00      1.00        14
     class_2       1.00      1.00      1.00         8

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36

